# 2. Filters: search, properties, components and detector types

This notebook works with two filter sets, `Generic/Johnson` (UBVRIJM) and `2MASS/2MASS` (J, H, Ks). It covers:
1. what an installed set contains;
2. NJM's catalogue: search, properties, descriptions, citations and plots;
3. components: building a curve from an instrument's own filter, optics and detector QE, with or without its site's atmosphere;
4. detector types (photon- or energy-counting): reading, tagging, correcting and converting;
5. combining sets for MESA;
6. the label rules a filter tree must keep (`label_check`).

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np

from sed_tools.api import Filters
from sed_tools.filters.curve_metadata import read_filter_header
from sed_tools.photometry import load_filter_curve

plt.rcParams["figure.dpi"] = 80
johnson = Filters.fetch("Generic", "Johnson")
twomass = Filters.fetch("2MASS", "2MASS")

## 1. What an installed set holds

Each `<band>.dat` is a CSV whose first row names the columns and carries the curve's metadata in a third field: `DetectorType`, its source, `WavelengthUnit`, and `Components` (what the published curve already includes). MESA reads the samples below it and loads the curves listed in the index file named after the folder.

In [ ]:
print(sorted(p.name for p in twomass.iterdir()))
print((twomass / "2MASS").read_text())
print((twomass / "Ks.dat").read_text().splitlines()[0])
print(read_filter_header(twomass / "Ks.dat"))

## 2. NJM's catalogue: search, properties, descriptions, citations, plots

NJM serves, beside the curves, `index.json` (every band), and per instrument `properties.json` (SVO-style characteristics computed from the curves) and `info.json` (descriptions, references, acknowledgements, licence). SVO is not needed for any of this.

In [ ]:
bands = Filters.search(wavelength=21500, fwhm=(1000, 5000))
print(len(bands), "bands cover 2.15 µm with a FWHM of 1000–5000 Å")
bands[["facility", "instrument", "band", "WavelengthPivot", "FWHM", "DetectorType"]].head(8)

In [ ]:
props = Filters.properties("2MASS", "2MASS")["bands"]
for band in ("J", "H", "Ks"):
    p = props[band]
    print(f"{band:3s} pivot {p['WavelengthPivot']:8.1f} Å  FWHM {p['FWHM']:7.1f} Å  "
          f"Vega ZP {p['ZeroPointVega_Jy']:7.1f} Jy  AB−Vega {p['ABminusVega']:.3f}")

In [ ]:
info = Filters.info("2MASS", "2MASS")
print(info["acknowledgements"][0]["text"][:200], "…")
print(Filters.cite("2MASS/2MASS", "Generic/Johnson/V")[:600], "…")

In [ ]:
ax = Filters.plot("Generic/Johnson", normalize="peak", vega=True)
ax.set_xlim(2500, 60000);

## 3. Components: filter, optics, QE and atmosphere

A published curve is often a product of several throughputs. `Filters.available` says what the published curve contains and which separate components the instrument's team published. 2MASS's published relative spectral responses include the atmosphere and the instrument; its filter, optics and NICMOS3 QE are also published separately (Explanatory Supplement III.1.b1).

In [ ]:
available = Filters.available("2MASS", "2MASS")
for key in ("published", "components", "buildable", "atmosphere_sites", "detector_type"):
    print(f"{key:18s} {available[key]}")

`Filters.fetch(..., components=[...])` builds a product folder under `products/<recipe>/`, which is itself a MESA instrument folder. A product with QE is photon-counting.
2MASS's atmosphere sites (PLEXUS models) state no airmass, so they are chosen with `airmass=None`.

In [ ]:
bare = Filters.fetch("2MASS", "2MASS", components=["filter", "optics", "qe"])
with_atm = Filters.fetch("2MASS", "2MASS", components=["filter", "optics", "qe"],
                         atmosphere={"site": "hopkins_2mass", "airmass": None})
print(read_filter_header(bare / "Ks.dat")["DetectorType"], read_filter_header(with_atm / "Ks.dat")["Components"])

Requests that cannot be met are refused with the reason, never approximated. Adding an atmosphere to the published curve, which already includes one, would count it twice:

In [ ]:
from sed_tools.filters.manifest import ComponentError

try:
    Filters.fetch("2MASS", "2MASS", components=["published"], atmosphere={"site": "hopkins_2mass", "airmass": None})
except ComponentError as refusal:
    print("Refused:", refusal)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for folder, label in ((twomass, "published (with atmosphere)"), (bare, "filter+optics+qe"),
                      (with_atm, "filter+optics+qe + hopkins_2mass")):
    curve = load_filter_curve(folder / "Ks.dat")
    ax.plot(curve.wavelength, curve.transmission / curve.transmission.max(), label=label)
ax.set(xlabel="Wavelength (Å)", ylabel="T / max", title="2MASS Ks", xlim=(18500, 24500))
ax.legend(fontsize=8);

## 4. Detector types

Synthetic photometry weights a curve by `T(λ)` for an energy-counting detector and by `T(λ)·λ` for a photon-counting one (Bessell & Murphy 2012, eq. A9). Every curve records which applies, and how that was established (`DetectorTypeSource`: `original` > `inferred` > `svo` > `svo-curve-match` > `assumed`). `FilterCurve.detector_type` says which.

In [ ]:
for path in sorted(johnson.glob("*.dat")) + sorted(twomass.glob("*.dat")):
    curve = load_filter_curve(path)
    print(f"{path.parent.name}/{path.stem:3s} {curve.detector_type:7s} {curve.metadata['detectortypesource']}")

`Filters.tag_detector_types(path, dry_run=True)` reports what tagging would write, without writing. Tagging resolves untagged files from SVO and never replaces a label with weaker evidence.

In [ ]:
rows = Filters.tag_detector_types(johnson, dry_run=True, verbose=False)
print(len(rows), "files;", sum(r["changed"] for r in rows), "would change")

A correction relabels curves (the numbers stay); a conversion also converts them: `T_E(λ) = T_P(λ)·λ`, rescaled to the same peak, so photometry is unchanged. Both edit files in place, so here they run on a copy made with `Filters.combine`.

In [ ]:
copy = Filters.combine("JohnsonCopy", "Generic/Johnson", facility="Custom", instrument="JohnsonCopy")
before = load_filter_curve(copy / "V.dat")
Filters.convert_detector_type(copy, "photon", bands=["V"])
after = load_filter_curve(copy / "V.dat")
print(before.detector_type, "->", after.detector_type, read_filter_header(copy / "V.dat").get("DetectorTypeConversion"))

# The converted curve has the same weight (T, or T·λ for photon counting), up to the peak rescaling.
def weight(curve):
    return curve.transmission * (curve.wavelength if curve.detector_type == "photon" else 1.0)

w_before = weight(before) / weight(before).max()
w_after = np.interp(before.wavelength, after.wavelength, weight(after)) / weight(after).max()
print("largest weight difference:", np.max(np.abs(w_before - w_after)))
assert np.allclose(w_before, w_after, atol=1e-6)

In [ ]:
Filters.correct_detector_type(copy, "photon", bands=["B"])
print(read_filter_header(copy / "B.dat")["DetectorType"], read_filter_header(copy / "B.dat")["DetectorTypeSource"])

## 5. Combining sets

`Filters.combine` (above) copies curves from several sets into one MESA instrument folder with its own index file, for example optical and near-infrared bands together.

In [ ]:
optical_ir = Filters.combine("JohnsonTwoMASS", "Generic/Johnson", "2MASS/2MASS",
                             facility="Custom", instrument="JohnsonTwoMASS")
print((optical_ir / "JohnsonTwoMASS").read_text())

## 6. Label rules: `label_check`

`sed_tools.filters.label_check` holds the rules every served filter tree must keep: known detector types and sources, known components, consistent labels for identical curves, a declared `WavelengthUnit`, and more (see its docstring). NJM runs it on every refresh. `problems(tree)` yields every violation; none means the tree passes.

This tree breaks one rule, because of section 4: `Custom/JohnsonCopy/B.dat` was relabelled photon-counting, while the same curve elsewhere in the tree (`Generic/Johnson/B.dat` and its copy in `Custom/JohnsonTwoMASS`) is energy-counting. Rule 7 says one curve must not be read in two conventions.

In [ ]:
from sed_tools.filters import label_check

problems = list(label_check.problems(Filters._filter_root))
for path, message in problems:
    print(path.relative_to(Filters._filter_root), "|", message)
assert problems and all("JohnsonCopy/B.dat" in message for _, message in problems)

## Command-line equivalents

| Python | Command line |
|---|---|
| `Filters.fetch("2MASS", "2MASS")` | `sed-tools filters 2MASS/2MASS` |
| `Filters.available(...)`, `Filters.info(...)`, `Filters.properties(...)` | `sed-tools filters-info 2MASS/2MASS` |
| `Filters.fetch(..., components=["filter", "optics", "qe"])` | `sed-tools filters-compose 2MASS/2MASS --components filter,optics,qe` |
| `atmosphere={"site": "hopkins_2mass", "airmass": None}` | `--atmosphere-site hopkins_2mass --airmass unstated` |
| `Filters.tag_detector_types(path, dry_run=True)` | `sed-tools filters-detector PATH --dry-run` |
| `Filters.convert_detector_type(path, "photon", bands=["V"])` | `sed-tools filters-detector PATH --convert photon --bands V` |
| `Filters.combine("X", "A/B", "C/D")` | `sed-tools filters-combine X A/B C/D` |
| `label_check.problems(tree)` | `python -m sed_tools.filters.label_check TREE [--against OLD_TREE]` (exits 1 when a rule is broken) |

`filters-detector` takes an installed set's name (`Facility/Instrument`) or a path (the filter root, an instrument folder or a file).

In [ ]:
!sed-tools filters-detector {johnson} --dry-run